<a href="https://colab.research.google.com/github/aadii-adnan/restaurant-ai-generator/blob/main/Menu_Gen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!pip install -qU langchain-groq

In [22]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [23]:
from langchain_groq import ChatGroq


llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)

# Testing the speed!
response = llm.invoke("Suggest a fancy name for Pakistani restaurant.")
print(response.content)

Here are a few suggestions for a fancy name for a Pakistani restaurant:

1. **Tandoor Mahal**: "Tandoor" refers to the traditional Pakistani clay oven, and "Mahal" means palace, conveying a sense of luxury and grandeur.
2. **Spice Route Karachi**: This name evokes the historic spice trade routes that connected Pakistan to the rest of the world, and "Karachi" adds a touch of authenticity.
3. **Dera Lahore**: "Dera" means "abode" or "homestead" in Urdu, and "Lahore" is a city in Pakistan known for its rich cultural heritage.
4. **Mughal Palace**: The Mughal Empire was a period of significant cultural and culinary achievement in Pakistani history, and "Palace" suggests a high-end dining experience.
5. **Khyber Grill**: The Khyber Pass is a famous mountain pass in Pakistan, and "Grill" highlights the restaurant's focus on traditional Pakistani barbecue dishes.
6. **Saffron & Spice**: Saffron is a prized spice in Pakistani cuisine, and "Spice" emphasizes the aromas and flavors of the restau

In [24]:
from langchain_core.prompts import PromptTemplate

In [25]:
# Use a new variable name: 'test_template' and 'test_chain'
test_template = PromptTemplate(
    input_variables=["cuisine"],
    template="Suggest a fancy name for a {cuisine} restaurant. DO NOT mention any other country."
)

test_chain = test_template | llm

# Force Mexican again
test_response = test_chain.invoke({"cuisine": "Mexican"})
print(test_response.content)

Here are some suggestions for a fancy name for a Mexican restaurant:

1. "Fiesta del Sol" (Party of the Sun)
2. "El Jardín de las Flores" (The Garden of Flowers)
3. "La Casa de las Delicias" (The House of Delights)
4. "Villa del Fuego" (Villa of Fire)
5. "Casa de las Estrellas" (House of the Stars)
6. "El Patio de las Olas" (The Patio of the Waves)
7. "La Hacienda de los Sueños" (The Estate of Dreams)
8. "Los Portales de la Vida" (The Gates of Life)

These names aim to evoke a sense of elegance, sophistication, and warmth, while also highlighting the vibrant culture and rich heritage of the restaurant's cuisine.


In [26]:
# This is our first "Station"
prompt_template_name = PromptTemplate(
    input_variables=['cuisine'],
    template="I want to open a restaurant for {cuisine} food. Suggest a fancy name for it."
)
name_chain = prompt_template_name | llm

In [27]:
# This is our second "Station"
# Note: The input here is 'restaurant_name', NOT 'cuisine'
prompt_template_items = PromptTemplate(
    input_variables=['restaurant_name'],
    template="Suggest some menu items for {restaurant_name}. Return them as a comma separated list."
)
menu_chain = prompt_template_items | llm

In [28]:
user_input = "Pakistani"
name_result = name_chain.invoke({"cuisine": user_input})
res_name = name_result.content

menu_result = menu_chain.invoke({"restaurant_name": res_name})
res_menu = menu_result.content

# 4. See the final result
print(f"Restaurant Name: {res_name}")
print(f"---")
print(f"Menu Items: {res_menu}")

Restaurant Name: For a fancy Pakistani restaurant, here are some name suggestions:

1. **Tandoor Mahal**: "Tandoor" refers to the traditional Pakistani clay oven, and "Mahal" means palace, conveying a sense of grandeur.
2. **Lahore Nights**: This name evokes the vibrant city of Lahore, known for its rich culinary heritage and cultural significance.
3. **Saffron & Spice**: Saffron is a luxurious spice commonly used in Pakistani cuisine, and "Spice" highlights the aromatic flavors of the food.
4. **Khyber's**: The Khyber Pass is a historic mountain pass in Pakistan, and using it as a namesake adds a touch of exoticism and adventure.
5. **Dastarkhwan**: This Urdu word means "tablecloth" or "feast," implying a lavish spread of delicious food.
6. **Mughal's Kitchen**: The Mughal Empire was a significant period in Pakistani history, and using this name pays homage to the region's rich culinary legacy.
7. **Bazaar Bites**: This name captures the essence of Pakistani street food and the bustli

In [29]:
prompt_template_name = PromptTemplate(
    input_variables=['cuisine'],
    template="I want to open a restaurant for {cuisine} food. Suggest a fancy name for it."
)

from langchain_core.runnables import RunnablePassthrough

chain1 = {"cuisine": RunnablePassthrough()} | prompt_template_name | llm

In [30]:
prompt_template_items = PromptTemplate(
    input_variables=['restaurant_name'],
    template="Suggest some menu items for {restaurant_name}. Return as a list."
)


full_chain = (
    {"restaurant_name": chain1}
    | RunnablePassthrough.assign(menu_items=prompt_template_items | llm)
)

result = full_chain.invoke("Mexican")

print(f"Name: {result['restaurant_name'].content}")
print(f"Menu: {result['menu_items'].content}")

Name: Here are some fancy name suggestions for your Mexican restaurant:

1. **El Jardín de las Flores** (The Garden of Flowers) - evoking a sense of vibrancy and beauty.
2. **Casa de las Olas** (House of Waves) - inspired by the ocean and the waves of Mexican culture.
3. **Azul y Sol** (Blue and Sun) - capturing the essence of Mexico's bright skies and turquoise waters.
4. **La Hacienda de los Abuelos** (The Grandparents' Estate) - conveying a sense of tradition and heritage.
5. **Fuego y Pasión** (Fire and Passion) - reflecting the bold flavors and fiery spirit of Mexican cuisine.
6. **El Patio de las Estrellas** (The Star Patio) - suggesting a magical and romantic atmosphere.
7. **Villa Mexicana** (Mexican Villa) - implying a luxurious and exotic experience.
8. **Los Amigos de la Cocina** (Friends of the Kitchen) - emphasizing the warm hospitality and delicious food.
9. **Cielo y Tierra** (Heaven and Earth) - representing the harmony between traditional and modern Mexican cuisine.
10

In [31]:
!pip install -q streamlit
!pip install -q pyngrok # Or you can use localtunnel

In [32]:
%%writefile app.py
import streamlit as st
import os
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

# 1. UI Setup
st.set_page_config(page_title="Restaurant AI", page_icon="🍴")
st.title("Restaurant Name & Menu Generator")

# 2. Sidebar for input
cuisine = st.sidebar.selectbox("Pick a Cuisine", ("Indian", "Mexican", "Italian", "Arabic", "Pakistani", "Chinese"))

# 3. Execution Logic (The Sequential Chain we built)
if cuisine:
    # Initialize the Brain
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)

    # Station 1: Name Generator
    prompt_template_name = PromptTemplate(
        input_variables=['cuisine'],
        template="I want to open a restaurant for {cuisine} food. Suggest a fancy name for it."
    )

    # Station 2: Menu Generator
    prompt_template_items = PromptTemplate(
        input_variables=['restaurant_name'],
        template="Suggest some menu items for {restaurant_name}. Return them as a bulleted list."
    )

    # Building the Assembly Line (Full Chain)
    chain1 = {"cuisine": RunnablePassthrough()} | prompt_template_name | llm

    full_chain = (
        {"restaurant_name": chain1}
        | RunnablePassthrough.assign(menu_items=prompt_template_items | llm)
    )

    # Run the machine!
    with st.spinner(f"Generating ideas for {cuisine} food..."):
        result = full_chain.invoke(cuisine)

    # 4. Display results on the Website
    # Logic: We use result['restaurant_name'].content to extract the text from the AI object
    st.header(result['restaurant_name'].content)

    st.subheader("Recommended Menu Items")
    st.write(result['menu_items'].content)

Overwriting app.py


In [33]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

--2026-05-01 19:26:41--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb [following]
--2026-05-01 19:26:41--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/ec689fe1-d727-4ebd-bbc3-5967730ab54e?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-01T20%3A17%3A04Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&

In [34]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

--2026-05-01 19:26:44--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb [following]
--2026-05-01 19:26:44--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/ec689fe1-d727-4ebd-bbc3-5967730ab54e?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-01T20%3A17%3A04Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&

In [35]:
import subprocess
import threading
import time

# Logic: Start Streamlit in a separate 'thread' so it doesn't block the rest of the code
def run_streamlit():
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

thread = threading.Thread(target=run_streamlit)
thread.start()

# Give Streamlit 3 seconds to warm up
time.sleep(3)

# Logic: Open the Cloudflare Tunnel
# This will output a link ending in .trycloudflare.com
!cloudflared tunnel --url http://localhost:8501

2026-05-01T19:26:49Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-01T19:26:49Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-01T19:26:51Z INF +--------------------------------------------------------------------------------------------+
2026-05-01T19:26:51Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-05-01T19:26:51Z INF |  https://groups-bidding-highways-bathroom.trycloudflar